Designing and implementing an LLM-powered chatbot and this chatbot will remember our previous interaction.

In [35]:
!pip install langchain_groq langchain_community

In [36]:
from posix import environ
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv('GROQ_API_KEY')
os.environ['Langchain_Tracing_v2']="True" #for tracing
os.environ['Langchain_Project']=os.getenv('LANGCHAIN_PROJECT')
os.environ['Langchain_api_key']=os.getenv('LANGCHAIN_API_KEY')

In [37]:
from langchain_groq import ChatGroq

model=ChatGroq(model='groq/compound-mini',groq_api_key=groq_api_key)
model

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x7f067bb102c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f067ba773b0>, model_name='groq/compound-mini', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [38]:
#conversing with our Model
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content='Hey ,This is masha and today i had a bad day csz my phone is not working ')])
# print(response)

AIMessage(content='Hey Masha, I’m sorry to hear you’re having a rough day and that your phone isn’t working. That can be really frustrating!  \n\nIf you’d like, I can walk you through some basic troubleshooting steps (like restarting, checking the battery, or looking at common software glitches). Just let me know what kind of phone you have and what’s happening (e.g., won’t turn on, screen is frozen, no signal, etc.), and we’ll try to get it sorted out. Hang in there!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 225, 'prompt_tokens': 477, 'total_tokens': 702, 'completion_time': 0.569572, 'completion_tokens_details': None, 'prompt_time': 0.025724, 'prompt_tokens_details': None, 'queue_time': 0.049146, 'total_time': 0.595295}, 'model_name': 'groq/compound-mini', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e06c3-9b2f-78e1-8dfd-10d316255169-0', tool_calls=[],

In [39]:
#validating if it remember our interaction or not
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content='Hey ,This is masha and today i had a bad day csz my phone is not working '),
        AIMessage(content='Hey Masha,I am really sorry to hear you are having a rough day—especially with a phone that’s not working.'),
        HumanMessage(content='hey , i forget why in was having a bad ..?')
    ]
)

AIMessage(content='Hey Masha,\n\nYou mentioned earlier that you were having a rough day because your phone isn’t working.\u202fIf you’re not sure exactly what went wrong, maybe we can figure it out together. Here are a few quick things to try, depending on what’s happening:\n\n| Symptom | Quick Checks & Fixes |\n|---------|----------------------|\n| **Phone won’t turn on** | 1. Plug it into a charger (use a wall outlet, not a USB hub). <br>2. Hold the power button for 10–15\u202fseconds (force‑restart). <br>3. Try a different charging cable or adapter. |\n| **Screen is black but you hear sounds** | 1. Connect to a computer and see if it’s recognized (helps confirm it’s alive). <br>2. If you have a removable battery, take it out for 30\u202fseconds, then re‑insert. |\n| **Stuck on logo / boot loop** | 1. Boot into recovery mode (usually Power\u202f+\u202fVolume‑Down or Power\u202f+\u202fVolume‑Up, depending on the model). <br>2. Choose “wipe cache partition” first; if that doesn’t help,

In [40]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
  if session_id not in store:
    store[session_id]=ChatMessageHistory()
  return store[session_id]


with_message_history=RunnableWithMessageHistory(model,get_session_history)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [41]:
config={'configurable':{'session_id':'chat1'}}

In [42]:
response=with_message_history.invoke(
    [HumanMessage(content='i  was having a bad day')],
    config=config
)

response.content

'I’m really sorry you’re having a rough day.\u202fIf you’d like to talk about what’s going on, I’m here to listen. Sometimes just sharing what’s bothering you can help lighten the load. If you’d rather get a quick pick‑me‑up, here are a few small things that might help:\n\n1. **Take a short break** – step outside for a few minutes, get some fresh air, or just stretch.\n2. **Do something kind for yourself** – a favorite snack, a quick walk, or a short video that makes you laugh.\n3. **Write it down** – jotting down what’s bothering you can give it a bit of distance and make it feel more manageable.\n4. **Reach out** – a quick text or call to a friend or family member can remind you that you’re not alone.\n\nIf you want to vent, need advice, or just want a distraction, let me know—I’m here for you. 🌼'

In [43]:
# changing the session_id
config1={'configurable':{'session_id':'chat2'}}
response=with_message_history.invoke(
    [HumanMessage(content='why i was having a bad day?')],
    config=config1

)
response.content

'I’m sorry you’re feeling that way. Bad days can happen for lots of different reasons, and often it’s a mix of things rather than one single cause. Here are some common factors that can make a day feel rough:\n\n| Possible cause | How it might show up | Quick check‑in |\n|----------------|----------------------|----------------|\n| **Lack of sleep** | Fatigue, irritability, trouble concentrating | Did you get 7‑9\u202fhours last night? |\n| **Stress or overload** | Racing thoughts, feeling overwhelmed, physical tension | Are you juggling many tasks or deadlines right now? |\n| **Conflict or miscommunication** | Arguments, feeling misunderstood, social tension | Did something happen with a friend, family member, or coworker? |\n| **Physical health** | Headaches, stomach upset, low energy | Have you eaten regularly, stayed hydrated, or taken any meds? |\n| **External events** | Bad weather, traffic, unexpected news | Did something outside your control happen today? |\n| **Unmet expectati

Let's inspect the `store` to verify if the chat history is being correctly maintained for each session ID (`chat1` and `chat2`).

In [48]:
print('History for chat1:')
for message in store['chat1'].messages:
    print(message)

print('\nHistory for chat2:')
for message in store['chat2'].messages:
    print(message)

History for chat1:

History for chat2:
